# ML09 · CNN 卷积神经网络

用卷积（convolution）与池化（pooling）让神经网络「看懂」图像，并亲手建一个小型卷积神经网络（convolutional neural network, CNN）分类 MNIST 手写数字。

## 学习目标
- 理解为何图像分类用 CNN 比多层感知器（multilayer perceptron, MLP）更合适。
- 创建卷积与池化的直觉：局部感受野（receptive field）、权重共享（weight sharing）、降采样（downsampling）。
- 分清楚信道（channel）与特征图（feature map）的概念与形状变化。
- 会用 torchvision 加载 MNIST 手写数字数据集（本书允许连网下载）。
- 能组装一个小型 CNN，完成训练并以准确率（accuracy）评估分类结果。

In [ ]:
# 概念：先做好绘图与乱数环境设置，后面所有范例共用
import numpy as np
import matplotlib.pyplot as plt

# 固定乱数种子，让每次运行造出的假数据与结果一致（方便对照讲解）
rng = np.random.default_rng(0)

# 让图以灰阶呈现，符合本书多为单信道图像的情境
plt.rcParams["image.cmap"] = "gray"

print("环境设置完成，numpy 版本:", np.__version__)

## 为何图像不用 MLP：CNN 的动机

图像是有空间结构（spatial structure）的数据：相邻像素（pixel）彼此高度相关，一条边、一个转角都是由「附近一小块」像素共同构成。

MLP 会先把图像摊平（flatten）成一长串矢量，这一步丢掉了「哪些像素互相相邻」的信息；而且全连接层会造成参数爆炸（parameter explosion），一张小图就需要极大量的权重。

CNN 改用「在图像上滑动的小滤镜」保留空间结构并共享权重，因此更省参数、也更稳定。

In [ ]:
# 概念：图像摊平成矢量后，相邻像素的关系（局部相关性 local correlation）就消失了
import numpy as np

# 造一张 4x4 的小灰阶方格图：左半边亮(1)、右半边暗(0)，中间有一条清楚的垂直边
image = np.array([
    [1, 1, 0, 0],
    [1, 1, 0, 0],
    [1, 1, 0, 0],
    [1, 1, 0, 0],
])

print("原图 shape:", image.shape, " 一眼可见第 1、2 栏之间有一条垂直边")
print(image)

flat = image.flatten()             # 摊平成一维矢量，模仿 MLP 的输入
print("\n摊平后 shape:", flat.shape)
print("摊平后:", flat)
# 注意：摊平后第 1 栏与第 2 栏原本相邻的像素，在矢量里被拆到不连续的位置，空间关系不见了

# 估算全连接的参数量：把这张图接到一个同样大小的隐藏层需要几个权重
n_pixels = image.size
fc_params = n_pixels * n_pixels    # 每个输入接每个输出，权重数是像素数的平方
print("\n全连接权重数（16->16）:", fc_params, "  图像一大，这个数字会爆炸")

## 卷积的直觉：滤镜与局部特征

卷积就是拿一个小滤镜（kernel / filter）在图像上滑动（sliding window），每滑到一个位置就做一次「局部加权相加」。

关键在于整张图用的是同一组滤镜权重（权重共享 weight sharing），所以同一个花样不管出现在哪里都能被侦测到。一个滤镜能看到的那一小块输入范围，称为感受野（receptive field）。

设计成「中间正、两侧负」的滤镜，就能做边缘侦测（edge detection）：在数值平坦处输出接近 0，在明暗交界处输出明显偏大。

In [ ]:
# 概念：手刻一个 3x3 边缘滤镜，用滑动窗口扫过整张图（这就是卷积）
import numpy as np

# 造一张 6x6 小图：左半亮、右半暗，中央一条垂直边
image = np.zeros((6, 6))
image[:, :3] = 1.0                 # 左半边设为亮

# 垂直边缘滤镜：左栏正、右栏负，遇到「左亮右暗」会得到大正值
kernel = np.array([
    [1.0, 0.0, -1.0],
    [1.0, 0.0, -1.0],
    [1.0, 0.0, -1.0],
])

k = kernel.shape[0]                # 滤镜边长（3）
out_h = image.shape[0] - k + 1     # 不补边时，输出每轴缩短 (k-1)
out_w = image.shape[1] - k + 1
feature_map = np.zeros((out_h, out_w))

for i in range(out_h):
    for j in range(out_w):
        patch = image[i:i + k, j:j + k]        # 取出滤镜当前覆盖的局部窗口
        feature_map[i, j] = np.sum(patch * kernel)  # 局部加权相加，得到这个位置的响应

print("原图:")
print(image)
print("\n特征图（边缘响应）shape:", feature_map.shape)
print(np.round(feature_map, 1))
# 技巧：绝对值大的位置就是被「点亮」的边；平坦区域接近 0
print("\n被侦测为边的最大响应:", feature_map.max())

## 信道与特征图：形状怎么变

信道（channel）是图像在「深度」方向的层数：灰阶图只有 1 个信道，彩色图有红绿蓝 3 个信道。

一个卷积层通常用多个滤镜，每个滤镜扫过输入后各自输出一张特征图（feature map）。有几个滤镜，输出就有几个信道。读懂 CNN 的关键，就是追踪张量形状 (信道, 高, 宽) 怎么随层变化。

形状：输入 (C_in, H, W) 经过 C_out 个滤镜，输出变成 (C_out, H', W')。

In [ ]:
# 概念：一张单信道图经过 2 个滤镜，会变成 2 张特征图（信道数 1 -> 2）
import numpy as np

# 造一张 1 信道、5x5 的小图，形状写成 (C, H, W) 以符合 CNN 惯例
image = rng.uniform(0, 1, size=(1, 5, 5))

# 准备 2 个不同的 3x3 滤镜：一个侦测垂直边、一个侦测水平边
vertical = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]], dtype=float)
horizontal = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]], dtype=float)
kernels = [vertical, horizontal]

def conv2d_single(channel, kernel):
    # 对单一信道做一次卷积，回传一张特征图
    k = kernel.shape[0]
    out_h = channel.shape[0] - k + 1
    out_w = channel.shape[1] - k + 1
    out = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            out[i, j] = np.sum(channel[i:i + k, j:j + k] * kernel)
    return out

single_channel = image[0]                 # 取出唯一的输入信道
feature_maps = [conv2d_single(single_channel, k) for k in kernels]
output = np.stack(feature_maps, axis=0)   # 把 2 张特征图叠成 (C_out, H', W')

print("输入形状 (C_in, H, W):", image.shape)
print("滤镜数量 (= 输出信道数):", len(kernels))
print("输出形状 (C_out, H', W'):", output.shape)
print("\n形状对照：信道 1 -> 2，空间 5x5 -> {}x{}".format(output.shape[1], output.shape[2]))

## 池化与降采样：抓重点、缩尺寸

池化（pooling）在特征图的小区块里只留一个代表值，借此降采样（downsampling）：缩小特征图、降低后续计算量。

最大池化（max pooling）取区块内最大值，保留「最强的响应」；平均池化（average pooling）取平均值。池化还带来平移不变性（translation invariance）：花样在小区块内轻微位移，输出仍大致相同。

形状：用 2x2、步幅 2 的池化，高与宽各约缩为一半。

In [ ]:
# 概念：对一张 4x4 特征图做 2x2 最大池化，比较池化前后的数值与尺寸
import numpy as np

# 造一张 4x4 的假特征图（数值代表各位置的响应强度）
feature_map = np.array([
    [1.0, 3.0, 2.0, 4.0],
    [5.0, 6.0, 1.0, 2.0],
    [7.0, 2.0, 9.0, 1.0],
    [0.0, 4.0, 3.0, 8.0],
])

pool = 2                              # 池化窗口边长，同时也是步幅 stride
out_h = feature_map.shape[0] // pool  # 不重叠地切，尺寸整除缩小
out_w = feature_map.shape[1] // pool
pooled = np.zeros((out_h, out_w))

for i in range(out_h):
    for j in range(out_w):
        block = feature_map[i * pool:(i + 1) * pool, j * pool:(j + 1) * pool]
        pooled[i, j] = block.max()    # 最大池化：只留区块内最强的响应

print("池化前 shape:", feature_map.shape)
print(feature_map)
print("\n池化后 shape:", pooled.shape, " 高与宽各缩为一半")
print(pooled)

## 加载 MNIST 数据集

MNIST 是经典的手写数字数据集，每张是 28x28 的灰阶图，标签（label）是 0 到 9。我们用 torchvision 下载它，这是本书唯一一次允许的连网下载。

图像需经过数据转换（transform）中的张量化（to-tensor），转成形状 (1, 28, 28) 的张量；再交给数据加载器（DataLoader）依批量（batch）分批喂入模型。

为什么用批量：一次处理一小批样本，比逐笔更快也更稳定。

In [ ]:
# 概念：用 torchvision 下载 MNIST，并把图像转成 (1, 28, 28) 张量
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ToTensor 会把 0-255 的像素缩到 0-1，并排成 (信道, 高, 宽) 的张量
transform = transforms.ToTensor()

# 注意：第一次运行会连网下载到 ./data，之后会直接读本机缓存
train_set = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_set = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# DataLoader 负责分批与打乱；训练要打乱、测试不需要
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)

image, label = train_set[0]            # 取出第一笔样本查看
print("训练样本数:", len(train_set), " 测试样本数:", len(test_set))
print("单张图像 shape:", tuple(image.shape), " 标签:", label)
print("像素值范围:", float(image.min()), "到", float(image.max()))

## 组一个小 CNN 并训练

把前面学到的零件接起来：两层「卷积 + 激活函数（activation, ReLU）+ 池化」负责抽特征，最后一层全连接层（fully connected layer）做分类。

分类用交叉熵损失（cross-entropy loss），直觉式为 loss = - Σ y · log(ŷ)：模型对正确类别给的几率越低，惩罚越大。训练循环（training loop）每一步都走「前向 -> 算损失 -> 反向传播（backpropagation）-> 更新权重」。

形状：(1, 28, 28) 经两次卷积与池化后缩小，摊平后接全连接层输出 10 个类别分数。

In [ ]:
# 概念：定义一个两层卷积 + 一层全连接的小 CNN，并走完整的训练循环
import torch
import torch.nn as nn

torch.manual_seed(0)                   # 固定权重初始化，让训练结果可重现

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 第一层卷积：输入 1 信道，输出 8 张特征图；padding=1 让空间尺寸不变
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)
        # 第二层卷积：信道数 8 -> 16，继续抽更抽象的特征
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2)   # 2x2 池化，每用一次空间尺寸减半
        self.relu = nn.ReLU()
        # 两次池化把 28x28 缩成 7x7；16 信道摊平后接到 10 个类别
        self.fc = nn.Linear(16 * 7 * 7, 10)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # 28x28 -> 14x14
        x = self.pool(self.relu(self.conv2(x)))   # 14x14 -> 7x7
        x = x.flatten(start_dim=1)                # 保留 batch 维，其余摊平成矢量
        return self.fc(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SmallCNN().to(device)
criterion = nn.CrossEntropyLoss()                 # 内含 softmax，直接吃类别分数
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_epochs = 2                                       # 为求快速示范只跑少数轮
for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()                      # 清掉上一步残留的梯度
        outputs = model(images)                    # 前向：得到 10 类分数
        loss = criterion(outputs, labels)          # 算交叉熵损失
        loss.backward()                            # 反向传播：算各权重的梯度
        optimizer.step()                           # 依梯度更新权重
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    print(f"epoch {epoch + 1}/{n_epochs}  平均训练损失: {avg_loss:.4f}")

## 评估准确率与看错在哪

训练损失只说明「在看过的数据上学得如何」，真正的表现要用没看过的测试集（test set）衡量。准确率（accuracy）就是预测（prediction）对的比例。

只看一个准确率数字不够：把模型「认错成什么」摊开来看，做错误分析（error analysis），更能找到改进方向。挑几张预测错的图，对照真实值与预测值，往往一眼就看出模型在哪里混淆（confusion）。

In [ ]:
# 概念：在测试集上算整体准确率，并挑几张预测错的图印出真实 vs 预测
import torch

model.eval()                                       # 切到评估模式（关掉只在训练用的行为）
correct, total = 0, 0
wrong_examples = []                                # 收集少数错例供查看

with torch.no_grad():                              # 评估不需算梯度，省内存也更快
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)              # 取分数最高的类别当预测
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        # 搜集前几个错例：真实与预测不同的样本
        if len(wrong_examples) < 4:
            mismatch = (preds != labels).nonzero(as_tuple=True)[0]
            for idx in mismatch:
                if len(wrong_examples) >= 4:
                    break
                wrong_examples.append((images[idx].cpu(), labels[idx].item(), preds[idx].item()))

accuracy = correct / total
print(f"测试集准确率: {accuracy:.4f}  （随机猜只有约 0.10）")

# 把错例画出来，标出真实值与预测值
fig, axes = plt.subplots(1, len(wrong_examples), figsize=(2.2 * len(wrong_examples), 2.4))
for ax, (img, true_label, pred_label) in zip(np.atleast_1d(axes), wrong_examples):
    ax.imshow(img.squeeze(0))                      # 去掉信道维才能当 2D 灰阶图画
    ax.set_title(f"true {true_label} / pred {pred_label}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 练习

以下三题由浅入深，皆为复合型但简短。数据请自己用 numpy 造（仿真即可），CNN 题沿用前面下载好的 MNIST。完成后对照「验收」确认输出。

In [ ]:
# TODO 1 ·（简单）楼层平面图的边缘侦测（集成：卷积 + 特征图）
#   情境：你有一张自造的小型建筑楼层平面格点图（墙=1、空地=0），想自动找出墙的边界。
#   要求：
#     1. 用 numpy 造一张 8x8 的平面图，把墙排成一个矩形外框（外圈为 1、内部为 0）。
#     2. 设计一个 3x3 边缘滤镜，对整张图做卷积，输出一张特征图。
#     3. 印出原图与特征图，并指出哪些位置被侦测为「边」（响应绝对值偏大）。
#   验收：应看到特征图在墙与空地的交界处数值明显偏高，墙内部与空地内部接近 0。
# TODO: 在这里完成


In [ ]:
# TODO 2 ·（中间）容积分布图的卷积加池化管线（集成：卷积 + 信道 + 池化 + 形状）
#   情境：一个城市街区被切成格网，每格记录自造的容积率（floor area ratio），
#         你想做一条「卷积 -> 池化」的特征管线并追踪形状变化。
#   要求：
#     1. 用 numpy 造一张 1 信道、16x16 的容积率格网（数值高低代表开发强度）。
#     2. 用 2 个不同滤镜做卷积，得到 2 张特征图，印出形状 (2, 高, 宽)。
#     3. 对这 2 张特征图各做 2x2 最大池化，再次印出形状，并用一句话说明尺寸为何减半。
#   验收：应看到信道数从 1 变 2、空间尺寸经池化后约缩为一半，形状变化前后对得起来。
# TODO: 在这里完成


In [ ]:
# TODO 3 ·（稍难）用小 CNN 分类建物缩略图并做错误分析（集成：MNIST 加载 + CNN 训练 + 准确率 + 错误查看）
#   情境：把 MNIST 手写数字当作「建物用途代码」的缩略图代理数据（数字 0-9 代表 10 种用途分区），
#         训练一个小 CNN 自动辨识代码。
#   要求：
#     1. 沿用前面下载的 MNIST，建一个含卷积、池化与全连接层的小 CNN，训练数个 epoch，
#        在测试集上计算整体准确率。
#     2. 统计测试集上「真实类别 -> 被预测成哪一类」，找出最常被认错的一对代码。
#        （技巧：可用一个 10x10 的计数表，列为真实、栏为预测，累加非对角线最大的格子。）
#     3. 挑 1-2 张该错误的缩略图印出真实值与预测值，并提出一句改进想法（如加一层卷积或增训练轮数）。
#   验收：应看到测试准确率明显高于随机猜（远高于 10%），
#         并能指认出一组最易混淆的类别与一张具体错例。
# TODO: 在这里完成


## 小结

- 图像有空间结构，MLP 把它摊平会丢掉相邻像素的关系且参数爆炸；CNN 用滑动的小滤镜保留结构并共享权重，更省更稳。
- 卷积是拿同一组滤镜权重在图像上滑动做局部加权，能侦测不管出现在哪的局部花样（如边缘）。
- 一个卷积层用多个滤镜，各产生一张特征图；读懂 CNN 的关键是追踪 (信道, 高, 宽) 的形状变化。
- 池化在小区块内只留代表值（最大池化取最大），缩小特征图、降低计算量，并带来对微小位移的不变性。
- torchvision 能加载 MNIST，图像被转成 (1, 28, 28) 张量并用 DataLoader 分批喂入模型。
- 一个小 CNN 由「卷积 + ReLU + 池化」堆栈加全连接层组成，训练走「前向 -> 算交叉熵损失 -> 反向 -> 更新」。
- 用没看过的测试集算准确率才公平；再做错误分析看模型把哪些类别认错，比单一数字更有洞见。